# 07 — LLMOps: Deney Takibi, A/B Test, Model Versiyonlama

**Kapsam:** LLMOps altyapısı — deney takibi, model versiyonlama, A/B test süreçleri ve
izleme panelleri.

Bu notebook üç bileşeni gösterir:
1. MLflow ile deney/eğitim takibi (04. ve 05. notebook'lardaki eğitimler zaten
   `report_to=["mlflow"]` ile otomatik loglanıyor)
2. İki model varyantını karşılaştıran istatistiksel A/B test
3. Streamlit izleme panelinin nasıl başlatılacağı

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## 1. MLflow deneylerini görüntüleme

Colab'da arka planda başlatıp `localtunnel` ile dışarı açıyoruz.

In [ ]:
get_ipython().system_raw("mlflow ui --backend-store-uri file:./mlruns --port 5000 &")
!npx --yes localtunnel --port 5000


## 2. A/B Test

Örnek: temel model vs. QLoRA+DPO modeli, aynı taslak-notlar seti üzerinde groundedness skoruna göre.

In [ ]:
from src.llmops.ab_test import run_ab_test
from src.rag.rag_pipeline import answer
from src.xai.hallucination_check import groundedness_score
from src.config import MODEL_CONFIG

eval_prompts = [
    "Aşağıdaki taslak notları bir README'ye dönüştür.\n\nTaslak notlar:\n- kurulum pip ile\n- kullanım örneği var",
    "Aşağıdaki taslak notları bir SSS bölümüne dönüştür.\n\nTaslak notlar:\n- şifre sıfırlama\n- veri saklama süresi",
    "Aşağıdaki taslak notları bir sorun giderme rehberine dönüştür.\n\nTaslak notlar:\n- port kullanımda hatası\n- ortam değişkeni eksik",
]

def eval_base(q):
    return answer(q, model_path=MODEL_CONFIG.base_llm)

def eval_finetuned(q):
    return answer(q, model_path="models/dpo-adapter")  # 05. notebook çıktısı

def score_fn(question, result):
    contexts = [c["text"] for c in result["contexts"]]
    return groundedness_score(result["answer"], contexts) if contexts else 0.0

result = run_ab_test("base", "finetuned", eval_base, eval_finetuned, eval_prompts, score_fn)
print(result)


## 3. İzleme paneli

FastAPI servisini (bkz. `src/serving/api.py`) çalıştırıp birkaç istek attıktan sonra:

In [ ]:
# Ayrı bir terminalde / hücrede:
# !uvicorn src.serving.api:app --host 0.0.0.0 --port 8000 &
# !streamlit run src/llmops/dashboard.py &
print("Panel için terminalde yukarıdaki komutları çalıştırın.")
